<a href="https://colab.research.google.com/github/tnkkzk0322/keiba-horse-profile-extractor/blob/main/%E7%AB%B6%E9%A6%ACID_%E7%B5%90%E6%9E%9C%E6%8A%BD%E5%87%BA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import random
import re
import time
import unicodedata
from typing import Dict, List, Optional, Tuple

import pandas as pd
import requests
from bs4 import BeautifulSoup


SEARCH_URLS = [
    "https://db.netkeiba.com//?pid=race_list&word=&start_year=2015&start_mon=none&end_year=none&end_mon=none&jyo%5B0%5D=30&jyo%5B1%5D=35&jyo%5B2%5D=36&jyo%5B3%5D=42&jyo%5B4%5D=43&jyo%5B5%5D=44&jyo%5B6%5D=45&jyo%5B7%5D=46&jyo%5B8%5D=47&jyo%5B9%5D=48&jyo%5B10%5D=50&jyo%5B11%5D=51&jyo%5B12%5D=54&jyo%5B13%5D=55&grade%5B0%5D=1&grade%5B1%5D=2&grade%5B2%5D=3&kyori_min=&kyori_max=&sort=date&list=20&page=1"
 ]

HEADERS = {
    "User-Agent": "MyResearchBot/1.0",
    "Accept-Language": "ja,en;q=0.8",
}

REQUEST_SLEEP_MIN = 5
REQUEST_SLEEP_MAX = 10
REQUEST_TIMEOUT_SEC = 30
RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}
MAX_FETCH_RETRIES = 2
DEBUG_FETCH = False

ENABLE_MEMORY_CACHE = True
HTML_CACHE: Dict[str, str] = {}
_last_request_at = 0.0

CENTRAL_RACECOURSES = {
    "札幌", "函館", "福島", "新潟", "東京",
    "中山", "中京", "京都", "阪神", "小倉",
}
LOCAL_RACECOURSES = {
    "門別", "盛岡", "水沢", "浦和", "船橋", "大井",
    "川崎", "金沢", "笠松", "名古屋", "園田", "姫路",
    "高知", "佐賀",
}
ALL_RACECOURSES = sorted(CENTRAL_RACECOURSES | LOCAL_RACECOURSES, key=len, reverse=True)

PAYOUT_KEYS = ["単勝", "馬連", "馬単", "3連複", "3連単"]


def fetch_log(*args) -> None:
    if DEBUG_FETCH:
        print("[FETCH]", *args)


def normalize_text(text: Optional[str]) -> str:
    if text is None:
        return ""
    text = text.replace("\xa0", " ").replace("\u3000", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def wait_for_polite_interval() -> None:
    global _last_request_at

    min_wait = random.uniform(REQUEST_SLEEP_MIN, REQUEST_SLEEP_MAX)
    now = time.monotonic()
    elapsed = now - _last_request_at

    if elapsed < min_wait:
        time.sleep(min_wait - elapsed)

    _last_request_at = time.monotonic()


def apply_best_effort_encoding(resp: requests.Response, url: str) -> requests.Response:
    content_type = (resp.headers.get("Content-Type") or "").lower()

    if "charset=euc-jp" in content_type:
        resp.encoding = "euc-jp"
        return resp
    if "charset=shift_jis" in content_type:
        resp.encoding = "shift_jis"
        return resp
    if "charset=utf-8" in content_type:
        resp.encoding = "utf-8"
        return resp

    head = resp.content[:3000].lower()
    if b"charset=euc-jp" in head:
        resp.encoding = "euc-jp"
    elif b"charset=shift_jis" in head:
        resp.encoding = "shift_jis"
    elif b"charset=utf-8" in head:
        resp.encoding = "utf-8"
    else:
        if "db.netkeiba.com" in url:
            resp.encoding = "euc-jp"
        else:
            resp.encoding = resp.apparent_encoding or "utf-8"

    return resp


def debug_response(resp: requests.Response, label: str = "") -> None:
    apply_best_effort_encoding(resp, resp.request.url)
    print("[FETCH]", "=" * 80)
    print("[FETCH]", label)
    print("[FETCH]", "request.method :", resp.request.method)
    print("[FETCH]", "request.url    :", resp.request.url)
    print("[FETCH]", "status_code    :", resp.status_code)
    print("[FETCH]", "final_url      :", resp.url)
    print("[FETCH]", "history        :", [(h.status_code, h.headers.get("Location")) for h in resp.history])
    print("[FETCH]", "server         :", resp.headers.get("Server"))
    print("[FETCH]", "content_type   :", resp.headers.get("Content-Type"))
    print("[FETCH]", "body_head      :", repr(resp.text[:200]))
    print("[FETCH]", "=" * 80)


def fetch_html(session: requests.Session, url: str) -> str:
    if ENABLE_MEMORY_CACHE and url in HTML_CACHE:
        return HTML_CACHE[url]

    last_exc = None

    for attempt in range(MAX_FETCH_RETRIES + 1):
        try:
            wait_for_polite_interval()

            res = session.get(
                url,
                timeout=REQUEST_TIMEOUT_SEC,
                allow_redirects=True,
            )
            apply_best_effort_encoding(res, url)

            if DEBUG_FETCH:
                debug_response(res, f"attempt={attempt} url={url}")

            if res.status_code == 200:
                html = res.text
                if ENABLE_MEMORY_CACHE:
                    HTML_CACHE[url] = html
                return html

            if res.status_code == 403:
                body_head = res.text[:200].replace("\n", " ")
                raise requests.HTTPError(
                    f"403 Forbidden for url: {url} | body_head={body_head!r}",
                    response=res,
                )

            if res.status_code in RETRYABLE_STATUS_CODES and attempt < MAX_FETCH_RETRIES:
                backoff = (2 ** attempt) + random.uniform(0.3, 0.8)
                fetch_log(f"retryable status={res.status_code}, sleep={backoff:.2f}s, url={url}")
                time.sleep(backoff)
                continue

            res.raise_for_status()

        except requests.HTTPError as e:
            last_exc = e
            status = getattr(getattr(e, "response", None), "status_code", None)

            if status == 403:
                raise

            if attempt < MAX_FETCH_RETRIES:
                backoff = (2 ** attempt) + random.uniform(0.3, 0.8)
                fetch_log(f"http retry sleep={backoff:.2f}s, error={repr(e)}")
                time.sleep(backoff)
                continue
            raise

        except (requests.ConnectionError, requests.Timeout) as e:
            last_exc = e
            if attempt < MAX_FETCH_RETRIES:
                backoff = (2 ** attempt) + random.uniform(0.3, 0.8)
                fetch_log(f"network retry sleep={backoff:.2f}s, error={repr(e)}")
                time.sleep(backoff)
                continue
            raise

        except Exception as e:
            last_exc = e
            raise

    if last_exc is not None:
        raise last_exc
    raise RuntimeError(f"fetch_html failed: {url}")


def get_soup(session: requests.Session, url: str) -> BeautifulSoup:
    html = fetch_html(session, url)
    return BeautifulSoup(html, "html.parser")


def replace_list_param(url: str, new_value: int) -> str:
    return re.sub(r"([?&]list=)\d+", rf"\g<1>{new_value}", url)


def get_total_count(soup: BeautifulSoup) -> Optional[int]:
    pager = soup.select_one("div.pager")
    if not pager:
        return None

    text = normalize_text(pager.get_text(" ", strip=True))
    m = re.search(r"(\d+)件中", text)
    return int(m.group(1)) if m else None


def normalize_grade(grade: str) -> str:
    g = grade.strip()
    mapping = {
        "GI": "G1",
        "GII": "G2",
        "GIII": "G3",
        "G1": "G1",
        "G2": "G2",
        "G3": "G3",
        "JpnI": "Jpn1",
        "JpnII": "Jpn2",
        "JpnIII": "Jpn3",
        "Jpn1": "Jpn1",
        "Jpn2": "Jpn2",
        "Jpn3": "Jpn3",
        "J・GI": "JG1",
        "J・GII": "JG2",
        "J・GIII": "JG3",
        "JG1": "JG1",
        "JG2": "JG2",
        "JG3": "JG3",
        "OP": "OP",
        "L": "L",
    }
    return mapping.get(g, g)


def clean_race_name_text(text: str) -> str:
    s = normalize_text(text)
    s = unicodedata.normalize("NFKC", s)

    s = re.sub(r"\s*出馬表\s*\|.*$", "", s)
    s = re.sub(r"\s*\|.*$", "", s)
    s = re.sub(r"\s*-\s*netkeiba.*$", "", s)
    s = re.sub(r"\s*[\(（]?\s*重賞\s*[\)）]?\s*", " ", s)

    return normalize_text(s)


def canonicalize_race_name(text: str, remove_kyoso: bool = False) -> str:
    s = clean_race_name_text(text)
    s = unicodedata.normalize("NFKC", s)

    s = re.sub(r"^第\s*\d+\s*回\s*", "", s)

    s = re.sub(
        r"\s*[\(（]\s*(?:G[1-3]|GI|GII|GIII|Jpn[1-3]|JpnI|JpnII|JpnIII|JG[1-3]|J・GI|J・GII|J・GIII)\s*[\)）]\s*$",
        "",
        s,
        flags=re.IGNORECASE,
    )

    s = re.sub(
        r"\s+(?:G[1-3]|GI|GII|GIII|Jpn[1-3]|JpnI|JpnII|JpnIII|JG[1-3]|J・GI|J・GII|J・GIII)\s*$",
        "",
        s,
        flags=re.IGNORECASE,
    )

    s = normalize_text(s)

    if remove_kyoso:
        s = re.sub(r"競走$", "", s).strip()

    return s if s else "レース名不明"


def split_race_name_and_grade(name: str) -> Tuple[str, str]:
    raw = normalize_text(name)
    m = re.search(r"\(([^()]+)\)\s*$", raw)

    if not m:
        return canonicalize_race_name(raw), "-"

    grade_raw = m.group(1)
    name_only = raw[:m.start()].strip()

    return canonicalize_race_name(name_only), normalize_grade(grade_raw)


def format_date_yyyy_mm_dd(text: str) -> str:
    text = normalize_text(text)

    m = re.search(r"(\d{4})/(\d{1,2})/(\d{1,2})", text)
    if m:
        y, mo, d = map(int, m.groups())
        return f"{y:04}/{mo:02}/{d:02}"

    m = re.search(r"(\d{4})年(\d{1,2})月(\d{1,2})日", text)
    if m:
        y, mo, d = map(int, m.groups())
        return f"{y:04}/{mo:02}/{d:02}"

    return text


def infer_racecourse(text: str) -> str:
    for course in ALL_RACECOURSES:
        if course in text:
            return course
    return "-"


def infer_holding_type(racecourse: str) -> str:
    if racecourse in CENTRAL_RACECOURSES:
        return "中央"
    if racecourse in LOCAL_RACECOURSES:
        return "地方"
    if racecourse != "-":
        return "海外"
    return "-"


def parse_course_distance(text: str) -> Tuple[str, str]:
    text = normalize_text(text)

    m = re.search(r"(芝|ダート|ダ|障害|障)\s*[左右内外直・ ]*\s*(\d{3,4})m?", text)
    if not m:
        return "-", "-"

    course_raw, distance = m.groups()
    course_map = {
        "芝": "芝",
        "ダ": "ダート",
        "ダート": "ダート",
        "障": "障害",
        "障害": "障害",
    }
    return course_map.get(course_raw, course_raw), distance


def parse_ground(text: str) -> str:
    text = normalize_text(text)

    m = re.search(r"(?:芝|ダート|ダ|障害|障)\s*:\s*(良|稍重|重|不良)", text)
    if m:
        return m.group(1)

    m = re.search(r"\b(良|稍重|重|不良)\b", text)
    return m.group(1) if m else "-"


def parse_age(text: str) -> str:
    text = normalize_text(text).replace("歳上", "歳以上")

    for pat in [r"(\d歳以上)", r"(\d歳)"]:
        m = re.search(pat, text)
        if m:
            return m.group(1)

    return "-"


def parse_sex(text: str) -> str:
    text = normalize_text(text)

    if "牡・牝・セ" in text or "牡牝セ" in text:
        return "牡・牝・セ"
    if "牡・牝" in text or "牡牝" in text:
        return "牡・牝"
    if "牡・セ" in text or "牡セ" in text:
        return "牡・セ"
    if "牝・セ" in text or "牝セ" in text:
        return "牝・セ"
    if "牝" in text and "牡" not in text and "セ" not in text:
        return "牝"
    if "牡" in text and "牝" not in text and "セ" not in text:
        return "牡"
    if "セ" in text and "牡" not in text and "牝" not in text:
        return "セ"

    return "-"


def parse_competition(text: str) -> str:
    text = normalize_text(text)

    for key in ["国際", "混合", "混", "特指", "九州産"]:
        if f"({key})" in text or key in text:
            return key
    return "-"


def parse_designation(text: str) -> str:
    text = normalize_text(text)

    for key in ["特指", "指", "市", "抽"]:
        if f"({key})" in text or key in text:
            return key
    return "-"


def parse_weight_rule(text: str) -> str:
    text = normalize_text(text)

    for key in ["定量", "別定", "ハンデ", "馬齢"]:
        if key in text:
            return key
    return "-"


def count_heads(soup: BeautifulSoup) -> str:
    table = soup.find("table", attrs={"summary": "全着順"})
    if not table:
        return "-"

    rows = table.find_all("tr")
    data_rows = [tr for tr in rows[1:] if tr.find_all("td")]
    return str(len(data_rows)) if data_rows else "-"


def extract_finishers(soup: BeautifulSoup) -> dict:
    result = {
        "1着": "-",
        "2着": "-",
        "3着": "-",
    }

    table = soup.find("table", attrs={"summary": "全着順"})
    if not table:
        return result

    rows = [tr for tr in table.find_all("tr")[1:] if tr.find_all("td")]

    rank_map = {"1": "1着", "2": "2着", "3": "3着"}
    found = set()

    for tr in rows:
        tds = tr.find_all("td")
        if not tds:
            continue

        rank_text = normalize_text(tds[0].get_text(" ", strip=True))
        if rank_text not in rank_map:
            continue

        horse_a = tr.select_one('a[href*="/horse/"]')
        horse_name = normalize_text(horse_a.get("title") or horse_a.get_text(" ", strip=True)) if horse_a else "-"

        result[rank_map[rank_text]] = horse_name or "-"
        found.add(rank_text)

        if found == {"1", "2", "3"}:
            break

    return result


def extract_payouts(soup: BeautifulSoup) -> dict:
    result = {
        "単勝": None,
        "馬連": None,
        "馬単": None,
        "3連複": None,
        "3連単": None,
    }

    label_map = {
        "単勝": "単勝",
        "馬連": "馬連",
        "馬単": "馬単",
        "三連複": "3連複",
        "三連単": "3連単",
        "3連複": "3連複",
        "3連単": "3連単",
    }

    tables = soup.find_all("table", attrs={"summary": "払い戻し"})
    if not tables:
        return result

    for table in tables:
        for tr in table.find_all("tr"):
            th = tr.find("th")
            tds = tr.find_all("td", recursive=False)

            if th is None or len(tds) < 2:
                continue

            raw_label = normalize_text(th.get_text(" ", strip=True))
            label = label_map.get(raw_label)
            if not label:
                continue

            amount_td = tds[1]

            nums = []
            for x in amount_td.stripped_strings:
                s = normalize_text(x)
                s = re.sub(r"[^\d]", "", s)
                if s != "":
                    nums.append(int(s))

            if nums:
                result[label] = nums[0]

    return result


def extract_race_ids_and_list_meta(soup: BeautifulSoup) -> Tuple[List[str], dict]:
    race_ids: List[str] = []
    seen = set()
    meta = {}

    table = soup.select_one("table.race_table_01")
    if not table:
        return race_ids, meta

    for tr in table.select("tr"):
        a = tr.select_one('td.w_race a[href^="/race/"]')
        if not a:
            continue

        href = a.get("href", "")
        m = re.search(r"^/race/(\d{12})/?$", href)
        if not m:
            continue

        race_id = m.group(1)
        if race_id not in seen:
            race_ids.append(race_id)
            seen.add(race_id)

        tds = tr.find_all("td")
        if len(tds) >= 9:
            raw_name = normalize_text(tds[4].get_text(" ", strip=True))
            name, grade = split_race_name_and_grade(raw_name)

            meta[race_id] = {
                "date": format_date_yyyy_mm_dd(tds[0].get_text(" ", strip=True)),
                "race_name": name,
                "grade": grade,
                "distance_src": normalize_text(tds[6].get_text(" ", strip=True)),
                "headcount": normalize_text(tds[7].get_text(" ", strip=True)),
                "ground": normalize_text(tds[8].get_text(" ", strip=True)),
                "1着": normalize_text(tds[11].get_text(" ", strip=True)) if len(tds) >= 12 else "-",
                "2着": normalize_text(tds[14].get_text(" ", strip=True)) if len(tds) >= 15 else "-",
                "3着": normalize_text(tds[15].get_text(" ", strip=True)) if len(tds) >= 16 else "-",
            }

    return race_ids, meta


def parse_race_detail(soup: BeautifulSoup, fallback: Optional[dict] = None) -> dict:
    fallback = fallback or {}

    h1 = soup.select_one("dl.racedata.fc h1")
    raw_name = normalize_text(h1.get_text(" ", strip=True)) if h1 else fallback.get("race_name", "-")
    race_name, grade = split_race_name_and_grade(raw_name)

    if grade == "-" and fallback.get("grade"):
        grade = fallback["grade"]

    p_tags = soup.select("div.data_intro p")
    p_texts = [normalize_text(p.get_text(" ", strip=True)) for p in p_tags]

    raw1 = p_texts[0] if len(p_texts) >= 1 else ""
    raw2 = p_texts[1] if len(p_texts) >= 2 else ""

    if not raw1:
        node = soup.select_one("div.RaceData01")
        raw1 = normalize_text(node.get_text(" ", strip=True)) if node else ""
    if not raw2:
        node = soup.select_one("div.RaceData02")
        raw2 = normalize_text(node.get_text(" ", strip=True)) if node else ""

    combined = normalize_text(" ".join([raw1, raw2]))

    date = format_date_yyyy_mm_dd(combined)
    if date == combined or date == "":
        date = fallback.get("date", "-")

    racecourse = infer_racecourse(combined)
    holding = infer_holding_type(racecourse)

    course, distance = parse_course_distance(raw1 if raw1 else fallback.get("distance_src", ""))
    if course == "-" or distance == "-":
        course2, distance2 = parse_course_distance(fallback.get("distance_src", ""))
        if course == "-":
            course = course2
        if distance == "-":
            distance = distance2

    ground = parse_ground(raw1)
    if ground == "-":
        ground = fallback.get("ground", "-")

    headcount = count_heads(soup)
    if headcount == "-":
        headcount = fallback.get("headcount", "-")

    finishers = extract_finishers(soup)
    payouts = extract_payouts(soup)

    if finishers["1着"] == "-" and fallback.get("1着"):
        finishers["1着"] = fallback["1着"]
    if finishers["2着"] == "-" and fallback.get("2着"):
        finishers["2着"] = fallback["2着"]
    if finishers["3着"] == "-" and fallback.get("3着"):
        finishers["3着"] = fallback["3着"]

    return {
        "レース名": race_name or fallback.get("race_name", "-"),
        "日付": date,
        "開催": holding,
        "性別": parse_sex(combined),
        "年齢": parse_age(combined),
        "競争": parse_competition(combined),
        "指定": parse_designation(combined),
        "重量": parse_weight_rule(combined),
        "頭数": headcount,
        "馬場": ground,
        "格": grade,
        "競馬場": racecourse,
        "コース": course,
        "距離": distance,
        "1着": finishers["1着"],
        "2着": finishers["2着"],
        "3着": finishers["3着"],
        "単勝": payouts["単勝"],
        "馬連": payouts["馬連"],
        "馬単": payouts["馬単"],
        "3連複": payouts["3連複"],
        "3連単": payouts["3連単"],
    }


def collect_race_ids_from_search_urls(
    session: requests.Session,
    search_urls: List[str],
) -> Tuple[List[str], dict]:
    merged_race_ids: List[str] = []
    merged_meta = {}
    seen = set()

    per_page = 20

    for search_url in search_urls:
        # まず1ページ目を取得して総件数を読む
        first_url = replace_list_param(search_url, per_page)
        first_url = replace_page_param(first_url, 1)

        first_soup = get_soup(session, first_url)
        total_count = get_total_count(first_soup)

        if total_count is None:
            raise RuntimeError(f"検索結果ページから総件数を取得できませんでした: {search_url}")

        total_pages = (total_count + per_page - 1) // per_page

        print(f"# total_count={total_count}")
        print(f"# total_pages={total_pages}")

        for page in range(1, total_pages + 1):
            page_url = replace_page_param(first_url, page)

            print(f"# fetching search page: {page}/{total_pages}")

            page_soup = get_soup(session, page_url)
            race_id_list, list_meta = extract_race_ids_and_list_meta(page_soup)

            print(f"# page={page}, race_ids={len(race_id_list)}")

            for race_id in race_id_list:
                if race_id not in seen:
                    merged_race_ids.append(race_id)
                    seen.add(race_id)

                if race_id not in merged_meta:
                    merged_meta[race_id] = list_meta.get(race_id, {})
                else:
                    for k, v in list_meta.get(race_id, {}).items():
                        if not merged_meta[race_id].get(k) and v:
                            merged_meta[race_id][k] = v

    return merged_race_ids, merged_meta


def clean_output_value(x) -> str:
    if x is None:
        return ""
    s = str(x)
    s = s.replace("\t", " ")
    s = s.replace("\r", " ")
    s = s.replace("\n", " ")
    s = s.replace("\xa0", " ")
    s = s.replace("\u3000", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def replace_page_param(url: str, new_page: int) -> str:
    if re.search(r"([?&]page=)\d+", url):
        return re.sub(r"([?&]page=)\d+", rf"\g<1>{new_page}", url)

    sep = "&" if "?" in url else "?"
    return f"{url}{sep}page={new_page}"


def build_session() -> requests.Session:
    session = requests.Session()
    session.headers.update(HEADERS)
    session.trust_env = False
    return session


def main() -> None:
    session = build_session()

    race_id_list, list_meta = collect_race_ids_from_search_urls(session, SEARCH_URLS)

    print("RACE_ID_LIST = [")
    for race_id in race_id_list:
        print(f'    "{race_id}",')
    print("]")

    rows = []
    for i, race_id in enumerate(race_id_list, start=1):
        detail_url = f"https://db.netkeiba.com/race/{race_id}/"
        print(f"# [{i}/{len(race_id_list)}] fetching detail: {detail_url}")
        try:
            soup = get_soup(session, detail_url)
            row = parse_race_detail(soup, fallback=list_meta.get(race_id, {}))
            rows.append(row)
        except Exception as e:
            print(f"# ERROR: race_id={race_id} の取得に失敗: {e}")

    df = pd.DataFrame(
        rows,
        columns=[
            "レース名", "日付", "開催", "性別", "年齢", "競争", "指定",
            "重量", "頭数", "馬場", "格", "競馬場", "コース", "距離",
            "1着", "2着", "3着", "単勝", "馬連", "馬単", "3連複", "3連単",
        ],
    )

    print()

    output_columns = [
        "レース名", "日付", "開催", "性別", "年齢", "競争", "指定",
        "重量", "頭数", "馬場", "格", "競馬場", "コース", "距離",
        "1着", "2着", "3着", "単勝", "馬連", "馬単", "3連複", "3連単",
    ]

    print("\t".join(output_columns))
    for _, row in df[output_columns].iterrows():
        values = [clean_output_value(row[col]) for col in output_columns]
        print("\t".join(values))


if __name__ == "__main__":
    main()


# total_count=466
# total_pages=24
# fetching search page: 1/24
# page=1, race_ids=20
# fetching search page: 2/24
# page=2, race_ids=20
# fetching search page: 3/24
# page=3, race_ids=20
# fetching search page: 4/24
# page=4, race_ids=20
# fetching search page: 5/24
# page=5, race_ids=20
# fetching search page: 6/24
# page=6, race_ids=20
# fetching search page: 7/24
# page=7, race_ids=20
# fetching search page: 8/24
# page=8, race_ids=20
# fetching search page: 9/24
# page=9, race_ids=20
# fetching search page: 10/24
# page=10, race_ids=20
# fetching search page: 11/24
# page=11, race_ids=20
# fetching search page: 12/24
# page=12, race_ids=20
# fetching search page: 13/24
# page=13, race_ids=20
# fetching search page: 14/24
# page=14, race_ids=20
# fetching search page: 15/24
# page=15, race_ids=20
# fetching search page: 16/24
# page=16, race_ids=20
# fetching search page: 17/24
# page=17, race_ids=20
# fetching search page: 18/24
# page=18, race_ids=20
# fetching search page: 19/24

In [ ]:
下記をお願いします。

## 対象
対象レース：新潟記念
対象年月：2008~2025
存在しない年はスルーしてOK

## 権限
osascriptのキー操作を許可する

## 事前条件
- ChromeでChatGPTにログイン済み
- ChatGPTの「競馬v2」プロジェクトが存在する
- Google Driveの `netkeiba_output` がローカル同期されている
- 添付対象ファイルは各年ごとに以下4つ
  - 出馬表.jsonl
  - 対戦表.jsonl
  - 過去10年傾向.jsonl
  - プロンプト.txt
- 0KBのjsonlは添付されなくても無視してOK

## 最初にやること
必要な権限を洗い出して、最初に私へ確認する
私がOKしたら、その後は確認なしで最後まで進める

## 注意事項
Chromeのショートカットは安定しないため、クリックで対応すること
**使用するモデルは毎回確認すること**

## 実行手順
各年について、以下を繰り返すこと。

1. ChromeのChatGPTを開く
   すでに開いていればそれを使う

2. 「競馬v2」プロジェクトを開く

3. プロジェクト内で新規チャットを始める

4. 必ず毎回、設定からモデルを `5.4` に変更する
   前回変更済みでも毎回変更が必要
   `Thinking` を選び、送信前に `5.4 Thinking` 相当になっていることを再確認する

5. ファイル添付ボタンを押す

6. Google Drive の `netkeiba_output` から対象年の4ファイルを開く
   可能なら4ファイル同時選択で添付する
   難しければ複数回に分けて添付してよい

7. 以下を確認する
   - jsonl 3つと txt 1つが添付されていること
   - 0KBのjsonlが添付されない場合は無視してよい

8. プロンプト欄に以下を入力する
   `{対象レース} {対象年}`
   例：`新潟大賞典 2008`

9. **送信直前にもう一度モデルが `5.4 Thinking` か確認する**

10. 送信する

11. その年の送信が終わったら、プロジェクトの新規チャットに戻って次の年を処理する

## 完了条件
- 2008~2025の各年について、存在する年をすべて送信し終えること
- 全件完了したら終了する
- 途中で逐一確認は不要。エラー時だけ止める

In [ ]:
3歳未勝利戦の考え方
・過去実績が同じ距離で好走している場合、今回も好走する可能性が高く、評価を上げること
・今回のレースが過去実績と異なる距離の場合、陣営の方向転換を期待し、評価を上げること
・過去実績が同じ距離で凡走し、今回も同距離の場合は評価を下げること